Cell 1: All imports and setup

In [1]:
import os
import torch
import torch.nn as nn
from torchvision import models, transforms
import cv2
from PIL import Image
import numpy as np
import gradio as gr
from huggingface_hub import InferenceClient
from dotenv import load_dotenv
import librosa
import librosa.display
import matplotlib.pyplot as plt

gr.close_all()

load_dotenv("D:/Deepfake-detection/.env")
HF_TOKEN = os.getenv("HF_TOKEN")

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)

client = InferenceClient(api_key=HF_TOKEN, provider="auto")
print("All imports ready.")

Using device: cuda
All imports ready.


Cell 2: Load all three models

In [2]:
IMG_SIZE = 224

image_transform = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

# --- Face/image/video model (fine-tuned version) ---
face_model = models.efficientnet_b0(weights=None)
face_model.classifier[1] = nn.Linear(face_model.classifier[1].in_features, 2)
face_model.load_state_dict(torch.load("D:/Deepfake-detection/saved_models/efficientnet_finetuned.pth", map_location=device))
face_model = face_model.to(device)
face_model.eval()
face_class_names = ['fake', 'real']

# --- Voice model (ASVspoof-trained) ---
voice_model = models.efficientnet_b0(weights=None)
voice_model.classifier[1] = nn.Linear(voice_model.classifier[1].in_features, 2)
voice_model.load_state_dict(torch.load("D:/Deepfake-detection/saved_models/asvspoof_efficientnet_best.pth", map_location=device))
voice_model = voice_model.to(device)
voice_model.eval()
voice_class_names = ['real', 'fake']  # bonafide=0, spoof=1

# --- Face detector ---
face_cascade = cv2.CascadeClassifier(cv2.data.haarcascades + 'haarcascade_frontalface_default.xml')

print("All models loaded successfully.")

All models loaded successfully.


Cell 3: Shared helper functions

In [3]:
def crop_face_from_frame(frame):
    gray = cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY)
    faces = face_cascade.detectMultiScale(gray, scaleFactor=1.1, minNeighbors=5, minSize=(60, 60))
    if len(faces) > 0:
        faces_sorted = sorted(faces, key=lambda f: f[2] * f[3], reverse=True)
        x, y, w, h = faces_sorted[0]
        return frame[y:y+h, x:x+w]
    return frame

def classify_face_crop(face_crop):
    face_rgb = cv2.cvtColor(face_crop, cv2.COLOR_BGR2RGB)
    pil_img = Image.fromarray(face_rgb)
    input_tensor = image_transform(pil_img).unsqueeze(0).to(device)
    with torch.no_grad():
        output = face_model(input_tensor)
        probs = torch.softmax(output, dim=1)[0]
    return probs.cpu().numpy()

def generate_explanation(prediction, confidence, fake_prob, real_prob):
    prompt = f"""Detection result: {prediction.upper()}
Confidence: {confidence}%
Fake probability: {fake_prob}%
Real probability: {real_prob}%

Write a short, 2-3 sentence explanation for the user about this deepfake
detection result. Be honest this is a statistical prediction, not certainty."""
    try:
        response = client.chat.completions.create(
            model="deepseek-ai/DeepSeek-V3-0324",
            messages=[{"role": "user", "content": prompt}],
            max_tokens=150
        )
        return response.choices[0].message.content
    except Exception:
        return "(Explanation unavailable right now.)"

Cell 4: The four prediction functions

In [4]:
# ---------------- IMAGE ----------------
def image_predict(pil_image):
    if pil_image is None:
        return "⚠️ Please upload an image."
    img_array = cv2.cvtColor(np.array(pil_image), cv2.COLOR_RGB2BGR)
    face_crop = crop_face_from_frame(img_array)
    probs = classify_face_crop(face_crop)
    pred_idx = int(np.argmax(probs))
    prediction = face_class_names[pred_idx]
    confidence = round(float(probs[pred_idx]) * 100, 2)
    fake_prob = round(float(probs[0]) * 100, 2)
    real_prob = round(float(probs[1]) * 100, 2)
    explanation = generate_explanation(prediction, confidence, fake_prob, real_prob)
    emoji = "🟢" if prediction == "real" else "🔴"
    return (f"{emoji} **{prediction.upper()}** — {confidence}% confidence\n\n"
            f"Fake: {fake_prob}%  |  Real: {real_prob}%\n\n"
            f"📝 {explanation}")

# ---------------- VIDEO ----------------
def video_predict(video_path):
    if video_path is None:
        return "⚠️ Please upload a video."
    cap = cv2.VideoCapture(video_path)
    total_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
    if total_frames <= 0:
        cap.release()
        return "❌ Could not read video."
    num_frames = 5
    frame_indices = [int(i * total_frames / num_frames) for i in range(num_frames)]
    all_probs = []
    for idx in frame_indices:
        cap.set(cv2.CAP_PROP_POS_FRAMES, idx)
        success, frame = cap.read()
        if not success:
            continue
        face_crop = crop_face_from_frame(frame)
        all_probs.append(classify_face_crop(face_crop))
    cap.release()
    if len(all_probs) == 0:
        return "❌ No frames could be processed."
    avg_probs = np.mean(all_probs, axis=0)
    pred_idx = int(np.argmax(avg_probs))
    prediction = face_class_names[pred_idx]
    confidence = round(float(avg_probs[pred_idx]) * 100, 2)
    fake_prob = round(float(avg_probs[0]) * 100, 2)
    real_prob = round(float(avg_probs[1]) * 100, 2)
    explanation = generate_explanation(prediction, confidence, fake_prob, real_prob)
    emoji = "🟢" if prediction == "real" else "🔴"
    return (f"{emoji} **{prediction.upper()}** — {confidence}% confidence "
            f"(averaged across {len(all_probs)} frames)\n\n"
            f"Fake: {fake_prob}%  |  Real: {real_prob}%\n\n"
            f"📝 {explanation}")

# ---------------- WEBCAM ----------------
def webcam_predict(pil_image):
    if pil_image is None:
        return "⚠️ Click the camera icon on the feed above to capture a photo, then Analyze."
    img_array = cv2.cvtColor(np.array(pil_image), cv2.COLOR_RGB2BGR)
    face_crop = crop_face_from_frame(img_array)
    probs = classify_face_crop(face_crop)
    pred_idx = int(np.argmax(probs))
    prediction = face_class_names[pred_idx]
    confidence = round(float(probs[pred_idx]) * 100, 2)
    emoji = "🟢" if prediction == "real" else "🔴"
    return f"{emoji} **{prediction.upper()}** — {confidence}% confidence"

# ---------------- VOICE ----------------
def voice_predict(audio_path):
    if audio_path is None:
        return "⚠️ Please upload or record audio."
    y, sr = librosa.load(audio_path, sr=16000)
    mel_spec = librosa.feature.melspectrogram(y=y, sr=sr, n_mels=128)
    mel_spec_db = librosa.power_to_db(mel_spec, ref=np.max)
    fig, ax = plt.subplots(figsize=(2.24, 2.24), dpi=100)
    ax.axis('off')
    librosa.display.specshow(mel_spec_db, sr=sr, ax=ax, cmap='magma')
    plt.subplots_adjust(left=0, right=1, top=1, bottom=0)
    temp_path = "D:/Deepfake-detection/outputs/temp_voice_spec.png"
    fig.savefig(temp_path, dpi=100)
    plt.close(fig)
    img = Image.open(temp_path).convert("RGB")
    input_tensor = image_transform(img).unsqueeze(0).to(device)
    with torch.no_grad():
        output = voice_model(input_tensor)
        probs = torch.softmax(output, dim=1)[0]
        pred_idx = torch.argmax(probs).item()
    prediction = voice_class_names[pred_idx]
    confidence = round(float(probs[pred_idx]) * 100, 2)
    real_prob = round(float(probs[0]) * 100, 2)
    fake_prob = round(float(probs[1]) * 100, 2)
    emoji = "🟢" if prediction == "real" else "🔴"
    return (f"{emoji} **{prediction.upper()}** — {confidence}% confidence\n\n"
            f"Real: {real_prob}%  |  Fake: {fake_prob}%\n\n"
            f"Trained on ASVspoof 2019 (89% accuracy on unseen synthesis methods).")

Cell 5: The polished UI

In [5]:
custom_theme = gr.themes.Soft(
    primary_hue="indigo",
    secondary_hue="slate",
).set(
    body_background_fill="*neutral_50",
    block_shadow="*shadow_drop_lg",
)

with gr.Blocks(title="Deepfake Detection System", theme=custom_theme) as demo:
    gr.Markdown(
        """
        # 🎭 AI-Powered Deepfake Detection System
        Detect manipulated media across **images**, **videos**, **live webcam**, and **voice audio** —
        powered by EfficientNet-B0 and explained in plain language by GenAI.
        """
    )

    with gr.Tabs():
        with gr.Tab("📷  Image"):
            with gr.Row():
                with gr.Column(scale=1):
                    img_input = gr.Image(type="pil", label="Upload an image")
                    img_button = gr.Button("🔍 Analyze Image", variant="primary")
                with gr.Column(scale=1):
                    img_output = gr.Markdown(label="Result")
            img_button.click(fn=image_predict, inputs=img_input, outputs=img_output)

        with gr.Tab("🎥  Video"):
            with gr.Row():
                with gr.Column(scale=1):
                    vid_input = gr.Video(label="Upload a video")
                    vid_button = gr.Button("🔍 Analyze Video", variant="primary")
                with gr.Column(scale=1):
                    vid_output = gr.Markdown(label="Result")
            vid_button.click(fn=video_predict, inputs=vid_input, outputs=vid_output)

        with gr.Tab("📹  Webcam"):
            gr.Markdown("Click the capture icon on the feed below, then Analyze.")
            with gr.Row():
                with gr.Column(scale=1):
                    webcam_input = gr.Image(type="pil", label="Webcam", sources=["webcam"])
                    with gr.Row():
                        snapshot_button = gr.Button("📸 Analyze Snapshot", variant="primary")
                        stop_button = gr.Button("🛑 Stop Camera", variant="stop")
                with gr.Column(scale=1):
                    webcam_output = gr.Markdown(label="Result")
            snapshot_button.click(fn=webcam_predict, inputs=webcam_input, outputs=webcam_output)
            stop_button.click(fn=lambda: None, inputs=None, outputs=webcam_input)

        with gr.Tab("🎙️  Voice"):
            gr.Markdown("Detects AI-synthesized/cloned speech (ASVspoof 2019, 89% accuracy on unseen synthesis methods).")
            with gr.Row():
                with gr.Column(scale=1):
                    voice_input = gr.Audio(type="filepath", label="Upload or record audio")
                    voice_button = gr.Button("🔍 Analyze Voice", variant="primary")
                with gr.Column(scale=1):
                    voice_output = gr.Markdown(label="Result")
            voice_button.click(fn=voice_predict, inputs=voice_input, outputs=voice_output)

    gr.Markdown("---\n*Built with PyTorch, EfficientNet-B0, OpenCV, and Hugging Face Inference Providers.*")

demo.launch()

C:\Users\ACER\AppData\Local\Temp\ipykernel_2416\3268104656.py:9: UserWarning: The parameters have been moved from the Blocks constructor to the launch() method in Gradio 6.0: theme. Please pass these parameters to launch() instead.
  with gr.Blocks(title="Deepfake Detection System", theme=custom_theme) as demo:


* Running on local URL:  http://127.0.0.1:7860
* To create a public link, set `share=True` in `launch()`.
